## Install required packages

```
pip install -r requirements.txt
```

## Check if you have Java

!java -version 

If you do not have java installed,
run 
```
sudo apt-get update
sudo apt-get install openjdk-8-jdk
```

#### macOS
[Here](https://mkyong.com/java/how-to-install-java-on-mac-osx/#homebrew-install-latest-java-on-macos) is the brief guide but note that if `/usr/local/Cellar` doesn't exist for you, follow the bellow steps

1. Install java `brew install java`
2. View java symbolic link instructions `brew info java`
3. Execute the command under "==> Caveats", (for me it's `sudo ln -sfn /opt/homebrew/opt/openjdk/libexec/openjdk.jdk /Library/Java/JavaVirtualMachines/openjdk.jdk`)
4. Validate that everything works with `java -version`

In [ ]:
import sys, os, subprocess

print("python:", sys.executable)
print("prefix:", sys.prefix)
print("VIRTUAL_ENV:", os.environ.get("VIRTUAL_ENV"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
subprocess.run(["java", "-version"])

## Working with Spark

First, load pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('SparkExample.com').getOrCreate()

# We load the csv into an RDD (Resilient Distributed Dataset) named tweetsCSV
sc = spark.sparkContext.textFile("training.1600000.processed.noemoticon.csv")
def process_string(s):
    split = s.split(',')
    if len(split) != 6:
        split[5] = ''.join(split[5:])
        split = split[:6]
    
    for i in range(6):
        split[i] = split[i][1:-1]
    return split
sc = sc.map(lambda s: process_string(s))

In [ ]:
# "take" the first 5 items
sc.take(5)

In [ ]:
deptColumns = ["target","ids", "date", "flag", "user", "text"]
# Create data frame
tweetsDF = sc.toDF(deptColumns)
tweetsDF.printSchema()
tweetsDF.show(truncate=True)

In [ ]:
# Select-Where query (selecting all the tweets of use "mimismo")
tweetsDF.where(tweetsDF.user=='starkissed').select(tweetsDF.date).collect()

In [ ]:
tweetsDF.createOrReplaceTempView("tweets")

### You can also use SQL queries 

In [ ]:

temp_df = spark.sql("SELECT * FROM tweets LIMIT 10")
temp_df.show()

## Introduction to NetworkX


In [ ]:
import networkx as nx

In [ ]:
G = nx.Graph()

G.add_node(234)
G.add_node("hello")
G.add_edge(234,"hello")

print("Nodes:", G.nodes())
print("Edges:", G.edges())



In [ ]:
G[234]

In [ ]:
G['hello']

In [ ]:
G.add_edge('Alice', 'Bob', know= 10, friends=5)

In [ ]:
print("Nodes:", G.nodes())
print("Edges:", G.edges())

In [ ]:
G['Bob']

In [ ]:
G['Alice']

In [ ]:
G['Bob']['Alice']['know'] += 1

In [ ]:
G['Alice']

## Analyzing graphs

In [ ]:
G.add_edge('Alice', 'Carlos')
G.add_edge('Carlos', 'Dave')
G.add_edge('Dave', 'Bob')
G.add_edge('Alice', 'Eve')

In [ ]:
components = nx.connected_components(G)
list(components)

In [ ]:
nx.degree(G)

In [ ]:
nx.degree(G,'Bob')

In [ ]:
nx.has_path(G, 'Alice', 'Dave')

In [ ]:
nx.has_path(G, 'Alice', 'hello')

In [ ]:
nx.shortest_path(G, 'Alice', 'Dave')

## Let's connect graphs and Twitter!

In [ ]:
import re
wordTokenizerRegex = re.compile("[ ,.;]")

# Split to words and flatten using flatMap (to have one big RDD with all words). Then, we filter only hashtags (starting with "#")
hashTags = sc.flatMap(lambda xs: [x.split(' ') for x in xs]).flatMap(lambda x: x).filter(lambda w: w.startswith("#"))

In [ ]:
# how many hashtags did we find
print(hashTags.count())

# examine the first 100
print(hashTags.take(100))

In [ ]:
# count unique words and sort them in descending order of count
countedHashTags = hashTags.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b).sortBy(lambda tup: tup[1], ascending = False)

In [ ]:
#show top 100 words
countedHashTags.take(100)

## Filtering tweets with selected hashtag #musicmonday

In [ ]:
temp_df = spark.sql("SELECT * FROM tweets WHERE LOWER(text) LIKE '%#musicmonday%'")
temp_df = temp_df.toPandas()
temp_df.head()

Once we have the required dataframe - we save it (to avoid having to re-run spark again next time)

In [ ]:
temp_df.to_csv("musicmonday.csv")

Now, let's construct a graph

In [ ]:
def addMentionedColumn(df):
    
    def mentionsList(txt):
        allWords = [word.strip(""" ,.:'\";""").lower() for word in txt.split()]
        allNames = [word.strip("@") for word in allWords if word.startswith("@")]
        uniqueNames = list(set(allNames))
        return uniqueNames
    
    df["mentioned"] = df["text"].apply(mentionsList)

In [ ]:
addMentionedColumn(temp_df)

In [ ]:
temp_df.head(10)

In [ ]:
def mentionGraph(df):
    g = nx.Graph()
    
    for  (index, target, ids, date, flag, user, text, mentionedUsers) in df.itertuples():
        for mentionedUser in mentionedUsers: 
            if (user in g) and (mentionedUser in g[user]):
                g[user][mentionedUser]["numberMentions"] += 1
            else:
                g.add_edge(user, mentionedUser, numberMentions=1)
    
    return g

In [ ]:
musicGraph = mentionGraph(temp_df)

In [ ]:
print("# nodes:", len(musicGraph.nodes()))
print("# edges:", len(musicGraph.edges()))

## Visualize Mention Graph

In [ ]:

from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from plotly.graph_objs import *

init_notebook_mode(connected=True)

In [ ]:
import random
def addRandomPositions(graph):
    posDict = dict((node,(random.gauss(0,10),random.gauss(0,10))) for node in graph.nodes())
    nx.set_node_attributes(graph, posDict, "pos")

In [ ]:
addRandomPositions(musicGraph)

In [ ]:
nx.get_node_attributes(musicGraph, 'pos')['acchanosaurus']

In [ ]:
def plotNetwork(graph):
    scatters=[]

    for (node1, node2) in graph.edges():
        x0, y0 = graph.nodes[node1]['pos']
        x1, y1 = graph.nodes[node2]['pos']
        edgeWidth = graph[node1][node2]['numberMentions']
        s = Scatter(
                x=[x0, x1],
                y=[y0, y1],
                hoverinfo='none',
                mode='lines', 
                line=Line(width=1 ,color='#888'))
        scatters.append(s)



    for node in graph.nodes():
        xPos, yPos = graph.nodes[node]['pos']
        s = Scatter(
                x=[xPos], 
                y=[yPos], 
                hoverinfo='none',
                mode='markers', 
                marker=dict(
                    color="#888", 
                    size=10,         
                    line=dict(width=2)))
        scatters.append(s)
    
    layout = Layout(showlegend=False)
    fig = Figure(data=scatters, layout=layout)
    iplot(fig, show_link=False )

In [ ]:
plotNetwork(musicGraph)